In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import json
import math

with open('move_to_id.json', 'r') as f:
    move_to_id = json.load(f)
PAD_ID = move_to_id["<PAD>"]

# Load the single tensor
encoded_tensor = torch.load('encoded_games_test.pt') # NEEDS TO BE CHANGED TO CORRECT FILE NAME

# Shift for X and Y
X_test = encoded_tensor[:, :-1]
Y_test = encoded_tensor[:, 1:]

test_dataset = TensorDataset(X_test, Y_test)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


FileNotFoundError: [Errno 2] No such file or directory: 'encoded_games_test.pt'

## Standard Decoder from "Attention is all you need" provided by pytorch implementation.

In [3]:
import torch.nn as nn

class ChessDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, max_len=200):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=1024)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # x shape: [batch, seq_len] → transformer expects [seq_len, batch]
        x = x.transpose(0, 1)
        seq_len, batch_size = x.size()

        # Add embeddings
        positions = torch.arange(seq_len, device=x.device).unsqueeze(1)
        x = self.embed(x) + self.pos_embed(positions)

        # Decoder masking: prevent attention to future tokens
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()

        x = self.decoder(x, x, tgt_mask=mask)
        logits = self.fc_out(x)  # [seq_len, batch, vocab_size]
        return logits.transpose(0, 1)  # [batch, seq_len, vocab_size]

In [ ]:
vocab_size = len(move_to_id)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'On device {device}')

with open("best_hparams_basic.json", "r") as f: 
    best_hparams = json.load(f)

params = best_hparams["best_params"]

# Rebuild model (same as in training)
model_basic = ChessDecoder(
    vocab_size=vocab_size,
    d_model=params["d_model"],
    nhead=params["nhead"],
    num_layers=params["num_layers"],
    max_len=200 # X_test.size(1)
).to(device)

# Load weights if you saved them
model_basic.load_state_dict(torch.load("basic_test.pt", map_location=device)) # NEEDS TO BE CHANGED TO CORRECT FILE NAME

model_basic.eval()


On device cuda


ChessDecoder(
  (embed): Embedding(11017, 512)
  (pos_embed): Embedding(200, 512)
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (multihead_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm3): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
  

## Expanded Attention Decoder

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        # Ensure that the model dimension (d_model) is divisible by the number of heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        # Initialize dimensions
        self.d_model = d_model # Model's dimension
        self.num_heads = num_heads # Number of attention heads
        self.d_k = d_model // num_heads # Dimension of each head's key, query, and value
        
        # Queries: one projection per head
        self.W_q = nn.Linear(d_model, d_model)
        # Shared keys and values across all heads
        self.W_k = nn.Linear(d_model, self.d_k)
        self.W_v = nn.Linear(d_model, self.d_k)

        # Output projection
        self.W_o = nn.Linear(d_model, d_model)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Calculate attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Apply mask if provided
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        # Softmax is applied to obtain attention probabilities
        attn_probs = torch.softmax(attn_scores, dim=-1)
        
        # Multiply by values to obtain the final output
        output = torch.matmul(attn_probs, V)
        return output
        
    def split_heads(self, x):
        # Reshape the input to have num_heads for multi-head attention
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        
    def combine_heads(self, x):
        # Combine the multiple heads back to original shape
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
        
    def forward(self, Q, K, V, mask=None):
        # Apply linear transformations and split heads
        # Multi-head queries
        Q = self.split_heads(self.W_q(Q))
        # Shared K, V
        K = self.W_k(K).unsqueeze(1)  # [B, 1, L, d_k]
        V = self.W_v(V).unsqueeze(1)  # [B, 1, L, d_k]
        
        # Perform scaled dot-product attention
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Combine heads and apply output transformation
        output = self.W_o(self.combine_heads(attn_output))
        return output
    
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))
    
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, tgt_mask):
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x
    
class ChessDecoderWithExpansion(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, max_len=200, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads=nhead, d_ff=1024, dropout=dropout) for _ in range(num_layers)])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)

        # Calculate positional embeddings
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        x = self.embed(x) + self.pos_embed(positions)

        # Decoder masking: prevent attention to future tokens
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()
        mask = mask.unsqueeze(0).unsqueeze(0)  # shape: [1, 1, seq_len, seq_len]


        for dec_layer in self.decoder_layers:
            x = dec_layer(x, mask)
    
        logits = self.fc_out(x)
        return logits  # [batch, seq_len, vocab_size]

In [ ]:
# Evaluation for expanded attention
vocab_size = len(move_to_id)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'On device {device}')

with open("best_hparams_expanded.json", "r") as f: 
    best_hparams = json.load(f)

params = best_hparams["best_params"]

# Rebuild model (same as in training)
model_expanded = ChessDecoderWithExpansion(
    vocab_size=vocab_size,
    d_model=params["d_model"],
    nhead=params["nhead"],
    num_layers=params["num_layers"],
    max_len=200 # X_test.size(1)
).to(device)

# Load weights if you saved them
model_expanded.load_state_dict(torch.load("basic_test.pt", map_location=device)) # NEEDS TO BE CHANGED TO CORRECT FILE NAME

model_expanded.eval()

On device cpu


NameError: name 'ChessDecoderWithExpandion' is not defined

In [ ]:
import torch.nn.functional as F

all_preds_basic = []
all_labels_basic = []
total_loss_basic = 0
total_tokens_basic = 0

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        logits = model_basic(x)

        loss = F.cross_entropy(
            logits.reshape(-1, vocab_size),
            y.reshape(-1),
            ignore_index=PAD_ID,
            reduction='sum'
        )
        total_loss_basic += loss.item()
        total_tokens_basic += (y != PAD_ID).sum().item()

        preds = torch.argmax(logits, dim=-1)

        # mask out padding so it doesn't affect metrics
        mask = (y != PAD_ID).reshape(-1)
        all_preds_basic.extend(preds.reshape(-1)[mask].cpu().numpy())
        all_labels_basic.extend(y.reshape(-1)[mask].cpu().numpy())

avg_loss_basic = total_loss_basic / total_tokens_basic

In [ ]:
# Expanded attention evaluation 
all_preds_expanded = []
all_labels_expanded = []
total_loss_expanded = 0
total_tokens_expanded = 0

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        logits = model_expanded(x)

        loss = F.cross_entropy(
            logits.reshape(-1, vocab_size),
            y.reshape(-1),
            ignore_index=PAD_ID,
            reduction='sum'
        )
        total_loss_basic += loss.item()
        total_tokens_basic += (y != PAD_ID).sum().item()

        preds = torch.argmax(logits, dim=-1)

        # mask out padding so it doesn't affect metrics
        mask = (y != PAD_ID).reshape(-1)
        all_preds_expanded.extend(preds.reshape(-1)[mask].cpu().numpy())
        all_labels_expanded.extend(y.reshape(-1)[mask].cpu().numpy())

avg_loss_expanded = total_loss_expanded / total_tokens_expanded

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

acc_basic = accuracy_score(all_labels_basic, all_preds_basic)
f1_basic = f1_score(all_labels_basic, all_preds_basic, average='macro')  # or 'weighted'
kappa_basic = cohen_kappa_score(all_labels_basic, all_preds_basic)

print(f"Test Loss Basic: {avg_loss_basic:.4f}")
print(f"Accuracy Basic: {acc_basic:.4f}")
print(f"F1 Score Basic: {f1_basic:.4f}")
print(f"Cohen's Kappa Basic: {kappa_basic:.4f}")

acc_expanded = accuracy_score(all_labels_expanded, all_preds_expanded)
f1_expanded = f1_score(all_labels_expanded, all_preds_expanded, average='macro')  # or 'weighted'
kappa_expanded = cohen_kappa_score(all_labels_expanded, all_preds_expanded)

print(f"Test Loss Basic: {avg_loss_expanded:.4f}")
print(f"Accuracy Basic: {acc_expanded:.4f}")
print(f"F1 Score Basic: {f1_expanded:.4f}")
print(f"Cohen's Kappa Basic: {kappa_expanded:.4f}")


Test Loss: 0.0019
Accuracy: 0.9996
F1 Score: 0.9969
Cohen's Kappa: 0.9996
